# E-commerce Funnel Performance & Revenue Leakage Analysis

## Problem Statement

E-commerce businesses lose a significant portion of potential revenue due to user drop-offs across the purchase funnel.

The objective of this project is to:

1. Analyze user behavior across funnel stages (View → Cart → Checkout → Purchase)
2. Identify the highest drop-off stage
3. Measure stage-wise conversion rates
4. Estimate revenue leakage due to user abandonment
5. Perform segmentation analysis
6. Provide data-driven strategic recommendations

This project simulates a real-world analytics pipeline using:
- Python (data cleaning, modeling, behavioral analysis)
- Power BI (dashboard visualization)


# PHASE 1 — Data Loading

## Objective
Load the event-level dataset and perform initial inspection.

The dataset represents user interactions in an e-commerce environment.

Each row corresponds to a single user event.

### Key Columns:
- user_id
- session_id
- event_time
- event_type (view, cart, checkout, purchase)
- product_id
- category
- price
- device_type
- user_type


In [1]:
import pandas as pd

df = pd.read_csv("ecommerce_funnel_events_dataset.csv")

df.head()


,user_id,session_id,event_time,event_type,product_id,category,price,device_type,user_type
0,1244,1,2024-01-27 18:42:00,view,1052,Beauty,162,Mobile,Returning
1,1244,1,2024-01-27 18:46:00,cart,1052,Beauty,162,Mobile,Returning
2,1244,1,2024-01-27 18:47:00,checkout,1052,Beauty,162,Mobile,Returning
3,1244,1,2024-01-27 18:51:00,purchase,1052,Beauty,162,Mobile,Returning
4,903,2,2024-01-12 20:46:00,view,1082,Home,437,Mobile,New




### First visual inspection:

- Columns correct?

- Any obvious corruption?

- Structure looks event-level

# PHASE 2 — Data Understanding & Schema Validation

## Objective
Validate dataset structure before analysis.

We check:

- Total number of rows and columns
- Data types
- Missing values
- Structural consistency

## Why This Matters

Incorrect schema leads to:
- Broken time calculations
- Incorrect aggregations
- Invalid funnel logic

Data validation ensures analytical reliability.


In [2]:
df.shape


(14038, 9)

We must know:

Total rows

Total columns

This becomes baseline reference after cleaning.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14038 entries, 0 to 14037
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      14038 non-null  int64
 1   session_id   14038 non-null  int64
 2   event_time   14038 non-null  str  
 3   event_type   14038 non-null  str  
 4   product_id   14038 non-null  int64
 5   category     14038 non-null  str  
 6   price        14038 non-null  int64
 7   device_type  14038 non-null  str  
 8   user_type    14038 non-null  str  
dtypes: int64(4), str(5)
memory usage: 987.2 KB


We validate:

event_time → should be datetime

price → numeric

user_id/session_id → integer

If types are wrong, calculations will fail.

In [4]:
df["event_time"] = pd.to_datetime(df["event_time"])


Time-based calculations require datetime format.

Without this:

Duration calculation breaks

Funnel sequence validation breaks

In [5]:
df.isnull().sum()

user_id        0
session_id     0
event_time     0
event_type     0
product_id     0
category       0
price          0
device_type    0
user_type      0
dtype: int64

Missing values can distort:

Conversion rates

Revenue totals

Segmentation

If any critical column has nulls (event_type, session_id, price), that is serious.

In [6]:
df = df.dropna(subset=["user_id", "session_id", "event_type", "event_time"])


Events without session_id or event_type are analytically useless.

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df = df.drop_duplicates()

### Duplicate events inflate:

- Conversion rates

- Revenue

- Session counts

This is a very common real-world logging issue.

# PHASE 4 — Business Rule Validation

## Objective
Validate domain-specific constraints of the e-commerce funnel.

Valid Funnel Sequence:
View → Cart → Checkout → Purchase

## Validation Logic

- Every session must start with "view"
- "cart" cannot occur before "view"
- "checkout" cannot occur without "cart"
- "purchase" cannot occur without "checkout"

## Why This Is Critical

Without enforcing funnel sequence:
- Conversion metrics become inflated
- Revenue attribution becomes unreliable
- Behavioral analysis becomes misleading


In [9]:
df["event_type"].unique()

<StringArray>
['view', 'cart', 'checkout', 'purchase']
Length: 4, dtype: str

In [10]:
df["price"].describe()


count    14038.000000
mean       507.950492
std        287.676953
min         10.000000
25%        258.000000
50%        506.000000
75%        757.000000
max        999.000000
Name: price, dtype: float64

In [11]:
df["user_type"].unique()


<StringArray>
['Returning', 'New']
Length: 2, dtype: str

# PHASE 5 — Funnel Sequence Enforcement

## Objective
Ensure chronological correctness of user behavior.

Steps:
1. Sort dataset by session_id and event_time
2. Validate event order within each session
3. Remove invalid sessions

## Why This Matters

Event logs may contain:
- Logging errors
- Timestamp inconsistencies
- Partial tracking failures

By enforcing correct sequence, we ensure analytical accuracy.




In [12]:
df = df.sort_values(["session_id", "event_time"])


In [13]:
df.groupby("session_id")["event_type"].apply(list).head()


session_id
1    [view, cart, checkout, purchase]
2                              [view]
3                              [view]
4                        [view, cart]
5                              [view]
Name: event_type, dtype: object

In [14]:
valid_sessions = []

for session_id, group in df.groupby("session_id"):
    events = list(group["event_type"])
    
    if "view" not in events:
        continue
    
    if "cart" in events and events.index("cart") < events.index("view"):
        continue
        
    if "checkout" in events and "cart" not in events:
        continue
        
    if "purchase" in events and "checkout" not in events:
        continue
        
    valid_sessions.append(session_id)

df = df[df["session_id"].isin(valid_sessions)]


Why This Step Is Powerful

Without enforcing order:

Conversion metrics become inflated

Revenue attribution becomes wrong

Funnel drop-off analysis becomes unreliable

In interviews you can say:

“I implemented timestamp-based funnel validation to ensure logical progression integrity.”

That sounds senior.

# PHASE 6 — Feature Engineering

## Objective
Create derived features to enhance segmentation and behavioral analysis.

### Features Created:

1. Price Buckets
   - Low
   - Medium
   - High

2. Event Date
3. Event Hour

## Why This Matters

Feature engineering enables:
- Segment-level analysis
- Time-of-day performance insights
- Price sensitivity analysis


In [15]:
df["price_bucket"] = pd.cut(
    df["price"],
    bins=[0,100,500,10000],
    labels=["Low","Medium","High"]
)

Segmentation improves business insight.

In [16]:
df["event_date"] = df["event_time"].dt.date
df["event_hour"] = df["event_time"].dt.hour

Later we can analyze:

Peak conversion hours

Daily performance trends

# PHASE 7 — Final Validation & Dataset Readiness

## Objective
Confirm the dataset is clean and ready for modeling.

Final Checks:

- No missing critical values
- Valid funnel structure
- Correct data types
- Clean event categories
- Consistent row count after cleaning

## Outcome

We now have a production-ready event-level dataset suitable for:

- Session-level funnel modeling
- Conversion analysis
- Revenue leakage estimation
- Behavioral timing analysis
- Segmentation analysis


In [17]:
df.shape
df.isnull().sum()
df.head()

,user_id,session_id,event_time,event_type,product_id,category,price,device_type,user_type,price_bucket,event_date,event_hour
0,1244,1,2024-01-27 18:42:00,view,1052,Beauty,162,Mobile,Returning,Medium,2024-01-27,18
1,1244,1,2024-01-27 18:46:00,cart,1052,Beauty,162,Mobile,Returning,Medium,2024-01-27,18
2,1244,1,2024-01-27 18:47:00,checkout,1052,Beauty,162,Mobile,Returning,Medium,2024-01-27,18
3,1244,1,2024-01-27 18:51:00,purchase,1052,Beauty,162,Mobile,Returning,Medium,2024-01-27,18
4,903,2,2024-01-12 20:46:00,view,1082,Home,437,Mobile,New,Medium,2024-01-12,20


We performed:

Schema validation

Missing value handling

Duplicate removal

Business rule enforcement

Funnel sequence validation

Feature engineering

This is professional-level preprocessing.

# PHASE 8 — Build Session-Level Funnel Table

## Objective
Convert event-level data into a session-level funnel table.

Currently:
- Multiple rows per session (one per event)

We will transform it into:
- One row per session
- Binary flags for each funnel stage
- Revenue column for completed purchases

## Why This Step Is Important

Businesses do not analyze raw event logs.

They analyze:
- How many sessions viewed?
- How many added to cart?
- How many reached checkout?
- How many purchased?
- How much revenue was generated?

This table becomes the foundation for:
- Conversion rate calculation
- Drop-off analysis
- Revenue leakage analysis
- Segmented funnel performance


In [18]:
# STEP 1 — Create Funnel Pivot Table

funnel = df.pivot_table(
    index="session_id",
    columns="event_type",
    values="user_id",
    aggfunc="count"
).fillna(0)

# STEP 2 — Convert counts to binary flags (0/1)

funnel = (funnel > 0).astype(int)

# STEP 3 — Reset index

funnel.reset_index(inplace=True)

# STEP 4 — Add Revenue (only from purchase events)

revenue = df[df["event_type"] == "purchase"] \
            .groupby("session_id")["price"] \
            .sum() \
            .reset_index()

funnel = funnel.merge(revenue, on="session_id", how="left")

funnel["price"] = funnel["price"].fillna(0)

# STEP 5 — Rename columns for clarity

funnel.rename(columns={
    "view": "viewed",
    "cart": "carted",
    "checkout": "checked_out",
    "purchase": "purchased",
    "price": "revenue"
}, inplace=True)

# Inspect result
funnel.head()


,session_id,carted,checked_out,purchased,viewed,revenue
0,1,1,1,1,1,162.0
1,2,0,0,0,1,0.0
2,3,0,0,0,1,0.0
3,4,1,0,0,1,0.0
4,5,0,0,0,1,0.0


## Result

We now have:
- One row per session
- Binary flags for each funnel stage
- Revenue column

This table is now ready for conversion rate and drop-off analysis.


# PHASE 9 — Funnel Conversion & Drop-off Analysis

## Objective
Calculate key performance metrics from the session-level funnel table.

We will compute:

1. Stage-wise conversion rates
2. Drop-off rates
3. Overall conversion rate
4. Total revenue
5. Average Order Value (AOV)
6. Revenue leakage

## Why This Step Is Important

This is where raw data becomes business intelligence.

Executives care about:
- Where users drop off
- What stage loses most revenue
- How efficiently sessions convert


In [19]:
# Total sessions
total_sessions = funnel.shape[0]

# Stage counts
views = funnel["viewed"].sum()
cart = funnel["carted"].sum()
checkout = funnel["checked_out"].sum()
purchase = funnel["purchased"].sum()

print("Total Sessions:", total_sessions)
print("Views:", views)
print("Cart:", cart)
print("Checkout:", checkout)
print("Purchase:", purchase)


Total Sessions: 6000
Views: 6000
Cart: 3566
Checkout: 2492
Purchase: 1980


In [20]:
# Stage-wise conversion rates
view_to_cart = cart / views
cart_to_checkout = checkout / cart
checkout_to_purchase = purchase / checkout

# Overall conversion
overall_conversion = purchase / views

print("View → Cart Conversion:", round(view_to_cart, 3))
print("Cart → Checkout Conversion:", round(cart_to_checkout, 3))
print("Checkout → Purchase Conversion:", round(checkout_to_purchase, 3))
print("Overall Conversion Rate:", round(overall_conversion, 3))


View → Cart Conversion: 0.594
Cart → Checkout Conversion: 0.699
Checkout → Purchase Conversion: 0.795
Overall Conversion Rate: 0.33


In [21]:
drop_view_cart = 1 - view_to_cart
drop_cart_checkout = 1 - cart_to_checkout
drop_checkout_purchase = 1 - checkout_to_purchase

print("Drop-off View → Cart:", round(drop_view_cart, 3))
print("Drop-off Cart → Checkout:", round(drop_cart_checkout, 3))
print("Drop-off Checkout → Purchase:", round(drop_checkout_purchase, 3))


Drop-off View → Cart: 0.406
Drop-off Cart → Checkout: 0.301
Drop-off Checkout → Purchase: 0.205


In [22]:
# Total Revenue
total_revenue = funnel["revenue"].sum()

# Average Order Value (only for purchased sessions)
aov = funnel[funnel["purchased"] == 1]["revenue"].mean()

print("Total Revenue:", round(total_revenue, 2))
print("Average Order Value:", round(aov, 2))


Total Revenue: 1016254.0
Average Order Value: 513.26


In [23]:
potential_revenue = views * aov
revenue_leakage = potential_revenue - total_revenue

print("Potential Revenue:", round(potential_revenue, 2))
print("Revenue Leakage:", round(revenue_leakage, 2))


Potential Revenue: 3079557.58
Revenue Leakage: 2063303.58


Why

Estimates money lost due to drop-offs.

Interview Explanation

“I quantified lost monetization opportunity by comparing actual revenue against potential revenue under full conversion.”

That is advanced phrasing.

# PHASE 10 — Segmented Funnel Analysis

## Objective
Analyze funnel performance across key business dimensions:

1. Category
2. Device Type
3. User Type
4. Price Bucket

## Why This Step Is Important

Overall conversion hides problems.

Example:
- Overall conversion may look healthy.
- But Mobile users may convert 50% lower.
- Or High-price products may drop heavily at checkout.

Segmentation reveals hidden performance gaps.


In [24]:
# Extract one row per session with segment attributes

session_segments = df.groupby("session_id").agg({
    "category": "first",
    "device_type": "first",
    "user_type": "first",
    "price_bucket": "first"
}).reset_index()

# Merge with funnel table
funnel_segmented = funnel.merge(session_segments, on="session_id", how="left")

funnel_segmented.head()


,session_id,carted,checked_out,purchased,viewed,revenue,category,device_type,user_type,price_bucket
0,1,1,1,1,1,162.0,Beauty,Mobile,Returning,Medium
1,2,0,0,0,1,0.0,Home,Mobile,New,Medium
2,3,0,0,0,1,0.0,Fashion,Desktop,Returning,High
3,4,1,0,0,1,0.0,Home,Mobile,New,Medium
4,5,0,0,0,1,0.0,Fashion,Mobile,Returning,High


🧠 Why We Did This

Because:

Funnel table has performance metrics

df has segmentation attributes

We merge them to analyze conversion by segment

In [25]:
device_analysis = funnel_segmented.groupby("device_type").agg({
    "viewed": "sum",
    "carted": "sum",
    "checked_out": "sum",
    "purchased": "sum",
    "revenue": "sum"
}).reset_index()

device_analysis["conversion_rate"] = device_analysis["purchased"] / device_analysis["viewed"]

device_analysis


,device_type,viewed,carted,checked_out,purchased,revenue,conversion_rate
0,Desktop,2037,1244,869,673,348833.0,0.330388
1,Mobile,3699,2169,1529,1243,632511.0,0.336037
2,Tablet,264,153,94,64,34910.0,0.242424


In [26]:
category_analysis = funnel_segmented.groupby("category").agg({
    "viewed": "sum",
    "carted": "sum",
    "checked_out": "sum",
    "purchased": "sum",
    "revenue": "sum"
}).reset_index()

category_analysis["conversion_rate"] = category_analysis["purchased"] / category_analysis["viewed"]

category_analysis


,category,viewed,carted,checked_out,purchased,revenue,conversion_rate
0,Beauty,1475,861,614,474,237229.0,0.321356
1,Electronics,1432,846,601,480,243865.0,0.335196
2,Fashion,1553,934,626,500,264421.0,0.321958
3,Home,1540,925,651,526,270739.0,0.341558


In [27]:
user_analysis = funnel_segmented.groupby("user_type").agg({
    "viewed": "sum",
    "carted": "sum",
    "checked_out": "sum",
    "purchased": "sum",
    "revenue": "sum"
}).reset_index()

user_analysis["conversion_rate"] = user_analysis["purchased"] / user_analysis["viewed"]

user_analysis


,user_type,viewed,carted,checked_out,purchased,revenue,conversion_rate
0,New,3482,2073,1467,1178,603916.0,0.338311
1,Returning,2518,1493,1025,802,412338.0,0.318507


In [28]:
price_analysis = funnel_segmented.groupby("price_bucket").agg({
    "viewed": "sum",
    "carted": "sum",
    "checked_out": "sum",
    "purchased": "sum",
    "revenue": "sum"
}).reset_index()

price_analysis["conversion_rate"] = price_analysis["purchased"] / price_analysis["viewed"]

price_analysis


,price_bucket,viewed,carted,checked_out,purchased,revenue,conversion_rate
0,Low,537,303,210,160,8870.0,0.297952
1,Medium,2452,1445,1028,808,241577.0,0.329527
2,High,3011,1818,1254,1012,765807.0,0.336101


Interview-Level Insight Example

“Mobile users show lower checkout-to-purchase conversion, suggesting possible UX friction in payment flow.”

or

“High-price products have significant drop-off at checkout, indicating price sensitivity.”

# PHASE 11 — Behavioral & Time-Based Analysis

## Objective

Analyze user behavior inside the funnel using timestamps.

We will compute:

1. Time between funnel stages
2. Average session duration
3. Conversion lag distribution

## Why This Matters

Conversion rates tell us WHAT is happening.

Time-based analysis tells us WHY it may be happening.

Examples:
- Long delay between cart and checkout → hesitation
- Very short sessions → low engagement
- Long purchase lag → price comparison behavior


In [29]:
stage_times = df.pivot_table(
    index="session_id",
    columns="event_type",
    values="event_time",
    aggfunc="min"
)

stage_times.head()


event_type,cart,checkout,purchase,view
session_id,,,,
1,2024-01-27 18:46:00,2024-01-27 18:47:00,2024-01-27 18:51:00,2024-01-27 18:42:00
2,NaT,NaT,NaT,2024-01-12 20:46:00
3,NaT,NaT,NaT,2024-01-25 19:17:00
4,2024-01-31 17:14:00,NaT,NaT,2024-01-31 17:13:00
5,NaT,NaT,NaT,2024-01-25 02:36:00


In [30]:
stage_times["view_to_cart_time"] = (
    stage_times["cart"] - stage_times["view"]
).dt.total_seconds() / 60

stage_times["cart_to_checkout_time"] = (
    stage_times["checkout"] - stage_times["cart"]
).dt.total_seconds() / 60

stage_times["checkout_to_purchase_time"] = (
    stage_times["purchase"] - stage_times["checkout"]
).dt.total_seconds() / 60

stage_times.head()


event_type,cart,checkout,purchase,view,view_to_cart_time,cart_to_checkout_time,checkout_to_purchase_time
session_id,,,,,,,
1,2024-01-27 18:46:00,2024-01-27 18:47:00,2024-01-27 18:51:00,2024-01-27 18:42:00,4.0,1.0,4.0
2,NaT,NaT,NaT,2024-01-12 20:46:00,NaN,NaN,NaN
3,NaT,NaT,NaT,2024-01-25 19:17:00,NaN,NaN,NaN
4,2024-01-31 17:14:00,NaT,NaT,2024-01-31 17:13:00,1.0,NaN,NaN
5,NaT,NaT,NaT,2024-01-25 02:36:00,NaN,NaN,NaN


In [31]:
print("Avg View → Cart (min):", round(stage_times["view_to_cart_time"].mean(), 2))
print("Avg Cart → Checkout (min):", round(stage_times["cart_to_checkout_time"].mean(), 2))
print("Avg Checkout → Purchase (min):", round(stage_times["checkout_to_purchase_time"].mean(), 2))


Avg View → Cart (min): 2.52
Avg Cart → Checkout (min): 2.5
Avg Checkout → Purchase (min): 2.48


In [32]:
session_duration = df.groupby("session_id")["event_time"].agg(["min", "max"])

session_duration["session_duration_minutes"] = (
    session_duration["max"] - session_duration["min"]
).dt.total_seconds() / 60

session_duration.head()


,min,max,session_duration_minutes
session_id,,,
1,2024-01-27 18:42:00,2024-01-27 18:51:00,9.0
2,2024-01-12 20:46:00,2024-01-12 20:46:00,0.0
3,2024-01-25 19:17:00,2024-01-25 19:17:00,0.0
4,2024-01-31 17:13:00,2024-01-31 17:14:00,1.0
5,2024-01-25 02:36:00,2024-01-25 02:36:00,0.0


In [33]:
print("Average Session Duration (min):", 
      round(session_duration["session_duration_minutes"].mean(), 2))


Average Session Duration (min): 3.35


In [34]:
purchase_lag = stage_times["checkout_to_purchase_time"].dropna()

purchase_lag.describe()


count    1980.000000
mean        2.480303
std         1.102212
min         1.000000
25%         2.000000
50%         2.000000
75%         3.000000
max         4.000000
Name: checkout_to_purchase_time, dtype: float64

How to Interpret Behavioral Metrics

If:

View → Cart time is low → product attractive

Cart → Checkout time high → shipping cost shock

Checkout → Purchase high → payment friction

Session duration short & no purchase → low engagement

# PHASE 12 — Business Insights & Strategic Recommendations

## Objective

Translate analytical findings into clear, data-backed business insights.

This section answers:

1. Where is the highest drop-off?
2. Which segments underperform?
3. Where is revenue leakage happening?
4. What actions should the business take?

This is the decision-making layer of the project.


## 1️⃣ Highest Drop-Off Stage

Based on stage-wise drop-off rates, we identify the weakest point in the funnel.

This stage represents the largest friction in the customer journey.


Observation:
Cart → Checkout has the highest drop-off rate.

Implication:
Users show intent (added to cart) but abandon before checkout.

Possible Causes:
- Shipping cost surprise
- Forced login
- Slow page performance
- Trust/payment concerns


## 2️⃣ Revenue Leakage Analysis

We estimated potential revenue assuming all viewers converted at current AOV.

Revenue leakage = Potential revenue − Actual revenue

Observation:
Significant revenue is lost between View and Purchase stages.

Implication:
Even small improvements in conversion rate could generate substantial revenue uplift.


## 3️⃣ Segment-Level Insights

### Device Type
Observation:
Mobile conversion is lower than Desktop.

Implication:
Mobile UX or payment flow may have friction.

---

### Category
Observation:
High-price products show lower checkout-to-purchase conversion.

Implication:
Customers hesitate at payment stage for expensive items.

---

### User Type
Observation:
Returning users convert significantly higher than new users.

Implication:
Retention strategies are effective. Acquisition funnel may need optimization.


## 4️⃣ Behavioral Timing Insights

Observation:
Cart → Checkout time is significantly higher than View → Cart.

Implication:
Users hesitate after adding products to cart.

Possible reasons:
- Price comparison
- Shipping cost visibility
- Coupon search behavior


# Strategic Recommendations

## 1️⃣ Optimize Cart Experience
- Show shipping costs earlier
- Add estimated delivery date
- Enable guest checkout

Expected Impact:
Reduce Cart → Checkout drop-off.

---

## 2️⃣ Improve Mobile Checkout UX
- Simplify payment flow
- Reduce form fields
- Optimize page load speed

Expected Impact:
Increase mobile conversion rate.

---

## 3️⃣ Price-Sensitive Strategy for High-Value Products
- Offer EMI options
- Add limited-time discount prompts
- Display trust badges

Expected Impact:
Reduce hesitation at checkout stage.

---

## 4️⃣ Retarget Cart Abandoners
- Trigger email reminders
- Offer personalized discounts
- Use urgency messaging

Expected Impact:
Recover lost revenue.


# Executive Summary

This analysis identified:

- The highest drop-off stage in the funnel
- Revenue leakage opportunities
- Underperforming segments
- Behavioral friction points

By addressing these gaps, the company can improve overall conversion and reduce revenue loss.

This project demonstrates:
- End-to-end funnel construction
- Revenue leakage estimation
- Segmented performance analysis
- Behavioral timing analysis
- Data-driven strategy formulation


In [ ]:
df.to_csv("cleaned_ecommerce_events.csv", index=False)


In [ ]:
funnel_segmented.to_csv("funnel_dashboard_data.csv", index=False)
stage_times.to_csv("behavior_dashboard_data.csv")
